In [90]:
import os
import base64
from mistralai.client import Mistral
from dotenv import load_dotenv

load_dotenv()

True

In [69]:
# Embedding direct avec Mistral
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
embedding = client.embeddings.create(model="mistral-embed", inputs=["test"])
vec = embedding.data[0].embedding

# Calcul de la norm
norm = sum(x**2 for x in vec) ** 0.5
print(f"Norm: {norm}")
print(f"First 5 values: {vec[:5]}")

Norm: 0.9999984440926123
First 5 values: [-0.021209716796875, 0.0294952392578125, 0.01474761962890625, 0.0228271484375, 0.03875732421875]


In [41]:
# OCR direct avec Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

MISTRAL_OCR_MODEL = "mistral-ocr-4-1"
file_path = "../data/pdf/1192.pdf"
file_name = "1192"


def encode_file(path: str) -> str | None:
    try:
        with open(path, "rb") as file:
            return base64.b64encode(file.read()).decode("utf-8")
    except Exception as error:
        print(f"Error while encoding {path}: {error}")
        return None


encoded_file = encode_file(file_path)
backoff = BackoffStrategy(
    initial_interval=5,
    max_interval=120,
    exponent=2.0,
    max_elapsed_time=300,  # Max 5 min
)

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
response = client.ocr.process(
    model=MISTRAL_OCR_MODEL,
    document={
        "type": "document_url",
        "document_url": f"data:application/pdf;base64,{encoded_file}",
        "document_name": file_name,
    },
    extract_footer=True,
    extract_header=True,
    table_format="markdown",
    retries=RetryConfig(strategy="backoff", backoff=backoff, retry_connection_errors=True),
)

In [42]:
import json

data = json.loads(response.json())

with open("../data/ocr/pdf/1192.json", "w", encoding="utf-8") as f:
    if isinstance(data, dict):
        json.dump(data, f, ensure_ascii=False, indent=4)

C:\Users\victor\AppData\Local\Temp\ipykernel_18184\4184321028.py:3: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  data = json.loads(response.json())


In [91]:
import re

MIN_CHUNK_LEN = 50
MAX_CHUNK_LEN = 5000


def split_text(text: str, max_chars: int) -> list[str]:
    if max_chars <= 0:
        raise ValueError("max_chars must be greater than zero")

    remaining = text.strip()
    chunks = []
    while len(remaining) > max_chars:
        split_at = remaining.rfind("\n", 0, max_chars + 1)
        if split_at < max_chars // 2:
            split_at = remaining.rfind(" ", 0, max_chars + 1)
        if split_at <= 0:
            split_at = max_chars

        chunks.append(remaining[:split_at].strip())
        remaining = remaining[split_at:].strip()

    if remaining:
        chunks.append(remaining)

    if len(chunks) > 1:
        print("Large text split into %s chunks (max_chars=%s)", len(chunks), max_chars)
        print(text)
    return chunks


def chunk_one_page(page: dict, document_metadata: dict) -> list[dict]:
    file_name = document_metadata["file_name"]
    file_name_no_ext = file_name.split(".")[0] if "." in file_name else file_name

    # mistral ocr 4 patterns
    table_pattern = re.compile(r"\[(tbl-[^]]+\.md)\]\([^)]+\.md\)")
    image_pattern = re.compile(r"!\[[^\]]+\.jpeg\]\([^)]+\.jpeg\)")
    superscript_pattern = re.compile(r"\$\^\{\d+\}\$")
    heading_pattern = re.compile(r"^(#{1,6})\s+(.*)")

    page_index = page.get("index", 0) + 1  # Start at 1
    markdown = page.get("markdown", "")

    if not markdown:
        print(f"[{file_name}] No markdown found for page={page_index} → skipping")
        return []

    chunks = []
    tables = {tbl["id"]: tbl for tbl in page.get("tables", [])}

    current_doc = ""
    section_title = ""
    section_level = 0
    section_index = 0
    chunk_index = 0
    has_table = False

    def flush_chunk():
        """Flush current buffer into a chunk"""
        nonlocal current_doc, chunk_index, section_title, section_level, has_table
        current_doc = current_doc.strip()
        if not current_doc:
            return

        if len(current_doc) < MIN_CHUNK_LEN:
            print(f"[{file_name}] Small chunk at page={page_index}, section={section_index} → skipping")
            print(f"chunk={current_doc}")
            return

        next_chunks = split_text(current_doc, MAX_CHUNK_LEN)
        for next_doc in next_chunks:
            chunk_index += 1
            chunks.append(
                {
                    "id": f"ssmesr_{file_name_no_ext}_p{page_index}_s{section_index}_c{chunk_index}",
                    "document": next_doc,
                    "metadata": {
                        **document_metadata,
                        "page_index": page_index,
                        "section_index": section_index,
                        "section_title": section_title[:200],
                        "section_level": section_level,
                        "chunk_len": len(current_doc.strip()),
                        "chunk_type": "paragraph",
                        "has_table": has_table,
                    },
                }
            )

        # Reset buffer
        current_doc = ""
        has_table = False

    for para in markdown.split("\n\n"):
        para = para.strip()
        if not para:
            continue

        lines = para.split("\n")
        first_line = lines[0]
        heading_match = heading_pattern.match(first_line)

        if heading_match:
            # New section: flush current chunk and reset
            flush_chunk()
            chunk_index = 0

            section_index += 1
            section_level = len(heading_match.group(1))
            title_parts = [heading_match.group(2)]

            # Collect multi-line section title
            for line in lines[1:]:
                line = line.strip()
                if line:
                    title_parts.append(line)
                else:
                    break
            section_title = " ".join(title_parts)

        else:

            # Check for tables in this paragraph
            if table_pattern.search(para):
                if len(para) > MAX_CHUNK_LEN:
                    print(f"[{file_name}] Large table at page={page_index}, section={section_index} → skipping")
                    # logger.debug(f"table={para}")
                    continue
                has_table = True

            # Process paragraph and add to chunk
            processed = para
            processed = table_pattern.sub(lambda m: tables.get(m.group(1), {}).get("content", ""), processed)
            processed = image_pattern.sub("", processed)
            processed = superscript_pattern.sub("", processed)
            processed = re.sub(r"\n{2,}", "\n\n", processed).strip()

            if not processed:
                continue

            # Try to add to current chunk
            next_doc = (current_doc + "\n\n" + processed) if current_doc else processed
            if len(next_doc) <= MAX_CHUNK_LEN:
                current_doc = next_doc
            else:
                # Chunk is full: flush and start new
                flush_chunk()
                current_doc = processed

    # Flush final chunk
    flush_chunk()
    return chunks

In [92]:
import json

with open("../data/ocr/pdf/1192.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [93]:
data

{'pages': [{'index': 0,
   'markdown': "# L'état\nde l'Emploi\nscientifique\nen France\n\nRapport\n2025",
   'images': [],
   'dimensions': {'dpi': 87, 'height': 1018, 'width': 720},
   'tables': [],
   'hyperlinks': [],
   'header': "MINISTÈRE CHARGÉ\nDE L'ENSEIGNEMENT\nSUPÉRIEUR\nET DE LA RECHERCHE\nLiberté\nÉgalité\nFraternité\nSIES",
   'footer': None},
  {'index': 1,
   'markdown': 'Le SIES, service statistique ministériel du MESR, produit des statistiques et des études sur l\'enseignement supérieur, l\'insertion professionnelle des étudiants et la recherche. Ses travaux constituent une aide à la décision utiles aux politiques publiques.\n\nCet ouvrage est édité par le SIES,\n\nSous-direction des systèmes d\'information et des études statistiques\n\nMinistère chargé de l\'Enseignement supérieur et de la Recherche (MESR)\n\n1 rue Descartes - 75231 Paris Cedex 05\n\nDirectrice de publication : Pierrette Schuhl\n\nRédacteur en chef : Louis Meuric\n\nExécution graphique : STDI\n\nDate

In [94]:
data.get("pages")[34].get("markdown").split("\n\n")

['## II.3',
 '# Le doctorat en France et dans le monde',
 "Les données des parties A à D exploitent l'enquête annuelle auprès des écoles doctorales et la source SISE$^{1}$. Ces deux sources utilisent deux nomenclatures de disciplines différentes (annexe) : respectivement la nomenclature des filières des doctorants et des étudiants en Master et la nomenclature des domaines scientifiques et groupes d'experts recherche (GER).",
 '## A Les doctorants',
 "L'enquête auprès des écoles doctorales menée par le MESR a recensé **69 600 étudiants inscrits durant l'année 2023-2024 (dénommée « rentrée 2023 » par commodité) pour préparer un diplôme national de doctorat**$^{2}$ ; ces effectifs sont en baisse de 1,5 % par rapport à la rentrée 2022, de 2,5 % par rapport à 2021 et de 9,4 % par rapport à 2013 (*tableau 11*).",
 "Sur la période 2013-2023, le nombre de doctorants diminue fortement en Sciences humaines et humanités (–21,0 %), ainsi qu'en Sciences de la société (–21,3 %) et en Sciences et tec

In [96]:
parsed = chunk_one_page(data.get("pages")[36], {"file_name": "coucou"})
parsed

[{'id': 'ssmesr_coucou_p37_s1_c1',
  'document': "en %\n\n|  Plus haut diplôme avant l'inscription en thèse | Rentrée universitaire |   |   |   |   |   | Évolution 2014-2023 en points  |\n| --- | --- | --- | --- | --- | --- | --- | --- |\n|   |  2014-15 | 2018-19 | 2019-20 | 2021-22 | 2022-23 | 2023-24  |   |\n|  **Doctorants inscrits avec un diplôme français :** | **78,7** | **77,8** | **77,9** | **81,3** | **81,5** | **80,7** | **2,0**  |\n|  Diplôme national de Master | 65,1 | 65,3 | 64,0 | 65,7 | 67,3 | 66,2 | 1,1  |\n|  Titre d'ingénieur, y compris double cursus Master ingénieur | 8,4 | 9,8 | 10,8 | 13,0 | 11,6 | 12,5 | 4,1  |\n|  Autre diplôme conférant le grade de Master, diplôme d'école de commerce ou de gestion | 5,2 | 2,7 | 3,0 | 2,5 | 2,6 | 2,0 | -3,2  |\n|  **Doctorants inscrits avec un diplôme étranger** | **21,3** | **22,2** | **22,1** | **18,7** | **18,5** | **19,3** | **-2,0**  |\n|  **Nombre de premières inscriptions en doctorat** | **17 262** | **16 039** | **16 374**